# Stage D — Keyword network and thematic matrix
## VOSviewer clusters, centrality, and the practical integration matrix

**AI adoption in urban planning governance: a systematic review**
Lartey & Law (2025), *Landscape and Urban Planning* 258, 105337

---

Sections 3.2.3, 3.2.4, 4.3 and 4.4. Two analyses:

- **Keyword co-occurrence network** (Figure 4, Table 1) — read from a VOSviewer JSON
  export. The clusters are recomputed here with centrality measures VOSviewer does not
  report, so the "less connected nodes" claims in Section 4.3 become numbers rather
  than visual impressions.
- **Practical integration matrix** (Figure 5c) — four themes against nine disciplines
  and application areas.

### On the matrix

The published matrix was populated with **qualitative ratings from content analysis**
(Section 3.2.3), which cannot be re-derived from metadata. Two versions are built:

1. **Published ratings**, reproduced as Figure 5c so the figure is regenerable.
2. **A corpus-derived matrix**, computed from term co-occurrence, shown beside it.

Comparing the two is the useful move: where they agree, the qualitative rating has
quantitative support; where they diverge, either the coding or the term lists deserve
a second look. Neither is presented as validating the other.

**Reads** `data/vosviewer/VOSviewer-network.json` and `data/corpus.csv`
**Writes** `outputs/figures/figure4_keyword_network.png`, `figure5c_matrix.png`,
`table1_clusters.csv`.

In [ ]:
from __future__ import annotations

import json
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


def _repo_relative(p):
    """Resolve whether the kernel started in notebooks/ or the repository root."""
    p = Path(p)
    if p.exists():
        return p
    alt = Path(str(p).replace("../", "", 1))
    return alt if (alt.exists() or Path("notebooks").is_dir()) else p


CONFIG = {"data_dir": "../data", "output_dir": "../outputs", "seed": 42, "dpi": 200}
SEED = CONFIG["seed"]
np.random.seed(SEED)

DATA_DIR = _repo_relative(CONFIG["data_dir"])
OUT_DIR = _repo_relative(CONFIG["output_dir"])
FIG_DIR, TAB_DIR = OUT_DIR / "figures", OUT_DIR / "tables"
for d in (FIG_DIR, TAB_DIR):
    d.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": CONFIG["dpi"],
                     "savefig.bbox": "tight", "font.size": 9,
                     "axes.titlesize": 10, "axes.titleweight": "bold"})

PALETTE = ["#9ecae1", "#6baed6", "#4292c6", "#2171b5", "#08519c"]

CORPUS_PATH = (DATA_DIR / "corpus_classified.csv"
               if (DATA_DIR / "corpus_classified.csv").exists()
               else DATA_DIR / "corpus.csv")
if not CORPUS_PATH.exists():
    raise FileNotFoundError(
        f"{CORPUS_PATH.resolve()} not found.\n"
        f"Run 01_corpus_assembly.ipynb first - it writes this file.")

corpus = pd.read_csv(CORPUS_PATH)
corpus["Year"] = pd.to_numeric(corpus["Year"], errors="coerce")
corpus = corpus.dropna(subset=["Year"])
corpus["Year"] = corpus["Year"].astype(int)

IS_DEMO = corpus["SourceFile"].astype(str).eq("SYNTHETIC").any() \
    if "SourceFile" in corpus.columns else False


def stamp(fig, demo=None, text="DEMO DATA\nNOT PAPER RESULTS"):
    """Mark synthetic output. `demo` defaults to the corpus, but a figure built from
    another source (e.g. the VOSviewer network) passes its own flag."""
    if (IS_DEMO if demo is None else demo):
        fig.text(0.5, 0.5, text, fontsize=42, color="grey", alpha=0.13,
                 ha="center", va="center", rotation=30, zorder=1000, weight="bold")
    return fig


def finish(fig, name, demo=None, stamp_text="DEMO DATA\nNOT PAPER RESULTS"):
    stamp(fig, demo=demo, text=stamp_text)
    path = FIG_DIR / f"{name}.png"
    fig.savefig(path)
    plt.show()
    print(f"  saved: {path}")


print(f"Corpus: {len(corpus):,} records, {corpus['Year'].min()}-{corpus['Year'].max()}")
print(f"Paper for comparison: 588 records across 83 countries")
if IS_DEMO:
    print("\n" + "!" * 66)
    print("This corpus is the SYNTHETIC demo set. Figures will be stamped.")
    print("!" * 66)

import networkx as nx

---
## D1 — Load the VOSviewer network

Export from VOSviewer with **File → Save → VOSviewer JSON file** and save it to
`data/vosviewer/VOSviewer-network.json`. Any co-occurrence map works; the loader reads
whatever weight and score attributes are present rather than assuming `Occurrences`
and `Avg. pub. year`.

If no file is found, a small synthetic network is generated so the rest of the
notebook runs.

In [ ]:
VOS_PATH = DATA_DIR / "vosviewer" / "VOSviewer-network.json"


def load_vosviewer(path):
    with open(path, "r", encoding="utf-8") as fh:
        net = json.load(fh)["network"]
    items = []
    for it in net.get("items", []):
        row = {"id": it.get("id"), "label": it.get("label"),
               "cluster": it.get("cluster"), "x": it.get("x"), "y": it.get("y")}
        for k, v in (it.get("weights") or {}).items():
            row[f"w_{k}"] = v
        for k, v in (it.get("scores") or {}).items():
            row[f"s_{k}"] = v
        items.append(row)
    items = pd.DataFrame(items)
    links = pd.DataFrame(net.get("links", []))
    if not links.empty and "strength" not in links.columns:
        links["strength"] = 1.0
    return items, links


def synth_network(n_terms=120, n_clusters=8, seed=SEED):
    rng = np.random.default_rng(seed)
    stems = ["ai", "machine learning", "smart city", "governance", "planning", "data",
             "policy", "mobility", "sustainability", "digital twin", "iot", "equity"]
    items = pd.DataFrame({
        "id": np.arange(1, n_terms + 1),
        "label": [f"{rng.choice(stems)} {i}" for i in range(n_terms)],
        "cluster": rng.integers(1, n_clusters + 1, n_terms),
        "x": rng.normal(0, 1, n_terms), "y": rng.normal(0, 1, n_terms),
        "w_Occurrences": rng.integers(2, 40, n_terms),
        "s_Avg. pub. year": rng.integers(2018, 2025, n_terms),
    })
    pairs = set()
    while len(pairs) < n_terms * 5:
        a, b = rng.integers(1, n_terms + 1, 2)
        if a != b:
            pairs.add((min(a, b), max(a, b)))
    links = pd.DataFrame([{"source_id": a, "target_id": b,
                           "strength": float(rng.integers(1, 6))} for a, b in pairs])
    return items, links


if VOS_PATH.exists():
    items, links = load_vosviewer(VOS_PATH)
    VOS_IS_DEMO = False
    print(f"Loaded {VOS_PATH.name}")
else:
    items, links = synth_network()
    VOS_IS_DEMO = True
    print("!" * 66)
    print(f"No VOSviewer export at {VOS_PATH}")
    print("Using a SYNTHETIC network so the notebook runs. Not real results.")
    print("Export from VOSviewer: File > Save > VOSviewer JSON file")
    print("!" * 66)

n_unlab = int(items["label"].isna().sum())
print(f"\nTerms: {len(items):,}   Links: {len(links):,}   Clusters: "
      f"{items['cluster'].nunique()}")
if n_unlab:
    print(f"  {n_unlab} item(s) have no label; kept in the graph, "
          f"excluded from term tables.")
display(items.head(5))

---
## D2 — Table 1: cluster profiles with centrality

VOSviewer reports cluster membership and occurrence counts. Section 4.3 goes further
and makes claims about *connectedness* — that terms like "AI regulation", "urban
health" and "equity" appear as smaller, less connected nodes. Degree and betweenness
make that measurable.

- **degree** — how many other terms it co-occurs with
- **weighted degree** — total co-occurrence strength
- **betweenness** — how often it sits on the shortest path between other terms; high
  values mark bridges between themes
- **eigenvector** — connection to other well-connected terms, i.e. centrality within
  the core rather than at the periphery

A term with high occurrences but low betweenness is frequent yet peripheral, which is
precisely the pattern Section 4.3 describes for the governance and equity terms.

In [ ]:
G = nx.Graph()
for r in items.itertuples():
    G.add_node(r.id, label=r.label, cluster=r.cluster)
for r in links.itertuples():
    G.add_edge(r.source_id, r.target_id, weight=float(r.strength))

deg = dict(G.degree())
wdeg = dict(G.degree(weight="weight"))
btw = nx.betweenness_centrality(G, seed=SEED)
try:
    eig = nx.eigenvector_centrality_numpy(G, weight="weight")
except Exception:
    eig = {n: np.nan for n in G}

metrics = pd.DataFrame({
    "id": list(G.nodes()),
    "label": [G.nodes[n].get("label") for n in G.nodes()],
    "cluster": [G.nodes[n].get("cluster") for n in G.nodes()],
    "degree": [deg[n] for n in G.nodes()],
    "weighted_degree": [wdeg[n] for n in G.nodes()],
    "betweenness": [round(btw[n], 5) for n in G.nodes()],
    "eigenvector": [round(eig.get(n, np.nan), 5) for n in G.nodes()],
}).sort_values("weighted_degree", ascending=False).reset_index(drop=True)

occ_col = next((c for c in items.columns if c.startswith("w_")), None)
yr_col = next((c for c in items.columns if c.startswith("s_")), None)

rows = []
for cl, grp in metrics.dropna(subset=["label"]).groupby("cluster"):
    src = items[items["cluster"] == cl]
    rows.append({
        "Cluster": int(cl),
        "Number of Items (Keywords)": len(grp),
        "Occurrences": int(src[occ_col].sum()) if occ_col else np.nan,
        "MeanYear": round(float(src[yr_col].mean()), 1) if yr_col else np.nan,
        "MeanBetweenness": round(float(grp["betweenness"].mean()), 5),
        "Example Nodes (Keywords)": ", ".join(
            t for t in grp.nlargest(5, "weighted_degree")["label"] if t),
    })
table1 = pd.DataFrame(rows).sort_values(
    "Number of Items (Keywords)", ascending=False).reset_index(drop=True)

print("Table 1 — cluster summary")
display(table1)
table1.to_csv(TAB_DIR / "table1_clusters.csv", index=False)

print("\nMost central terms overall:")
display(metrics.dropna(subset=["label"]).head(15))
metrics.to_csv(TAB_DIR / "network_metrics.csv", index=False)

# Frequent but peripheral: high occurrences, low betweenness
if occ_col:
    m = metrics.merge(items[["id", occ_col]], on="id", how="left").dropna(subset=["label"])
    m = m[m[occ_col] >= m[occ_col].median()]
    peripheral = m.nsmallest(12, "betweenness")[["label", "cluster", occ_col,
                                                 "degree", "betweenness"]]
    print("\nFrequent but peripheral terms (>= median occurrences, lowest betweenness).")
    print("Section 4.3 makes claims of exactly this shape about governance and equity terms:")
    display(peripheral)

---
## D3 — Figure 4: the network

Panel a is the co-occurrence map, drawn at VOSviewer's own coordinates so it is
comparable to the published figure, with node size by occurrence and colour by
cluster. Panel b colours the same layout by mean publication year, which is the
temporal overlay Section 4.3 uses to date the shift from "machine learning" toward
"AI regulation" and "digital twins".

In [ ]:
lab = items.dropna(subset=["label"]).copy()
pos = {r.id: (r.x, r.y) for r in items.itertuples()
       if pd.notna(r.x) and pd.notna(r.y)}
if len(pos) < len(items):
    pos = nx.spring_layout(G, seed=SEED, k=0.35, iterations=80)

fig, axs = plt.subplots(1, 2, figsize=(19, 9))

sizes = (lab[occ_col] if occ_col else pd.Series(5, index=lab.index)).astype(float)
sizes = np.clip(sizes / sizes.max() * 900, 18, 900)
top = lab.nlargest(12, occ_col) if occ_col else lab.head(12)

for ax, colour_by, title, cmap in [
        (axs[0], lab["cluster"].astype(float), "a) Keyword co-occurrence, by cluster", "tab20"),
        (axs[1], lab[yr_col].astype(float) if yr_col else lab["cluster"].astype(float),
         "b) Same layout, by mean publication year", "viridis")]:
    for u, v, d in G.edges(data=True):
        if u in pos and v in pos:
            ax.plot([pos[u][0], pos[v][0]], [pos[u][1], pos[v][1]],
                    color="#90A4AE", lw=0.2 + 0.9 * d["weight"] / links["strength"].max(),
                    alpha=0.4, zorder=1)
    xs = [pos[i][0] for i in lab["id"] if i in pos]
    ys = [pos[i][1] for i in lab["id"] if i in pos]
    keep = [i in pos for i in lab["id"]]
    sc = ax.scatter(xs, ys, s=sizes[keep], c=colour_by[keep], cmap=cmap,
                    alpha=0.85, edgecolor="white", linewidth=0.5, zorder=3)
    # Alternate the offset so the dense core stays legible
    for k, r in enumerate(top.itertuples()):
        if r.id in pos:
            ax.annotate(r.label, pos[r.id], fontsize=7.5, ha="center", zorder=5,
                        xytext=(0, 9 if k % 2 == 0 else -15),
                        textcoords="offset points",
                        bbox=dict(boxstyle="round,pad=0.15", fc="white",
                                  ec="none", alpha=0.7))
    if yr_col and ax is axs[1]:
        plt.colorbar(sc, ax=ax, label="Mean publication year", fraction=0.03)
    ax.set(title=title); ax.axis("off")

fig.suptitle("Figure 4 — Keyword co-occurrence network and temporal overlay",
             fontsize=13, weight="bold")
fig.tight_layout()
# This figure comes from the VOSviewer export, so its provenance is VOS_IS_DEMO -
# a real network read alongside a demo corpus must not be stamped as demo.
finish(fig, "figure4_keyword_network", demo=VOS_IS_DEMO,
       stamp_text="SYNTHETIC NETWORK")

---
## D4 — Figure 5c: the practical integration matrix

Four themes against nine disciplines and application areas, rated 1 (weak),
2 (moderate) or 3 (strong).

The left panel reproduces the published qualitative ratings. The right panel computes
the same grid from the corpus: for each theme–column pair, the share of records
matching both sets of terms, discretised into the same 1–3 bands by tercile.

The right panel is a different instrument measuring a related thing — term
co-occurrence is not the same as a coder's judgement of depth of coverage. Read the
difference map as a prompt for where to look, not as a score.

In [ ]:
MATRIX_THEMES = {
    "AI in Urban Design and Sustainable Development":
        ["urban design", "sustainab", "urban form", "green", "climate"],
    "AI in Decision-Making and Urban Management":
        ["decision-making", "decision making", "urban management", "decision support"],
    "Applications of AI and ML in Smart Cities":
        ["smart city", "smart cities", "machine learning", "iot", "sensor"],
    "Ethical and Social Implications":
        ["ethic", "equity", "privacy", "bias", "transparen", "social implication",
         "accountab", "trust"],
}

MATRIX_COLUMNS = {
    "Eng.":         ["engineering", "control system", "optimi"],
    "Comp. Sci.":   ["computer science", "algorithm", "computing", "software"],
    "Pub. Admin.":  ["public administration", "government", "public sector"],
    "Urban Stud.":  ["urban studies", "urbanism", "urban planning"],
    "Soc. Sci.":    ["social", "society", "sociolog", "community"],
    "Env. Plan.":   ["environment", "ecolog", "climate", "sustainab"],
    "Smart Cities": ["smart city", "smart cities", "digital twin"],
    "Transport":    ["transport", "mobility", "traffic"],
    "Land Use":     ["land use", "land-use", "zoning"],
}

# Published ratings, Figure 5c
PAPER_MATRIX = pd.DataFrame(
    [[2, 3, 2, 1, 1, 2, 3, 1, 1],
     [1, 3, 2, 2, 2, 2, 2, 1, 1],
     [2, 3, 1, 1, 1, 2, 3, 2, 2],
     [1, 2, 2, 1, 3, 1, 2, 1, 1]],
    index=list(MATRIX_THEMES), columns=list(MATRIX_COLUMNS))

blob = pd.Series("", index=corpus.index)
for f in ("Title", "Abstract", "Keyword", "Journal"):
    if f in corpus.columns:
        blob = blob + " " + corpus[f].fillna("").astype(str)
blob = blob.str.lower()


def hits(terms):
    return blob.str.contains("|".join(re.escape(t) for t in terms), regex=True)


theme_hits = {t: hits(v) for t, v in MATRIX_THEMES.items()}
col_hits = {c: hits(v) for c, v in MATRIX_COLUMNS.items()}

raw = pd.DataFrame({c: {t: int((theme_hits[t] & col_hits[c]).sum())
                        for t in MATRIX_THEMES} for c in MATRIX_COLUMNS})
raw = raw.loc[list(MATRIX_THEMES), list(MATRIX_COLUMNS)]

flat = raw.to_numpy().ravel()
if flat.max() > 0:
    q1, q2 = np.quantile(flat[flat > 0], [1 / 3, 2 / 3]) if (flat > 0).sum() > 2 else (0, 0)
    derived = raw.map(lambda v: 3 if v > q2 else (2 if v > q1 else 1))
else:
    derived = raw.map(lambda v: 1)

fig, axs = plt.subplots(1, 3, figsize=(21, 5.2))
cmap = sns.color_palette(["#DEEBF7", "#9ECAE1", "#2171B5"], as_cmap=False)

for ax, data, title, kw in [
        (axs[0], PAPER_MATRIX, "Published ratings (Figure 5c)", dict(vmin=1, vmax=3)),
        (axs[1], derived, "Derived from your corpus", dict(vmin=1, vmax=3))]:
    sns.heatmap(data, annot=True, fmt="d", cmap=cmap, cbar=False,
                linewidths=1.2, linecolor="white", ax=ax,
                annot_kws={"weight": "bold"}, **kw)
    ax.set(title=title, xlabel="", ylabel="")
    ax.set_yticklabels([l.get_text()[:42] for l in ax.get_yticklabels()],
                       rotation=0, fontsize=7.5)
    ax.tick_params(axis="x", rotation=45)

diff = derived - PAPER_MATRIX
sns.heatmap(diff, annot=True, fmt="+d", cmap="RdBu_r", center=0, vmin=-2, vmax=2,
            linewidths=1.2, linecolor="white", ax=axs[2],
            cbar_kws={"label": "derived − published"})
axs[2].set(title="Difference", xlabel="", ylabel="")
axs[2].set_yticklabels([]); axs[2].tick_params(axis="x", rotation=45)

handles = [plt.Rectangle((0, 0), 1, 1, facecolor=c) for c in
           ["#DEEBF7", "#9ECAE1", "#2171B5"]]
axs[0].legend(handles, ["1 Weak", "2 Moderate", "3 Strong"], fontsize=7.5,
              loc="upper left", bbox_to_anchor=(0, -0.32), ncol=3, frameon=False)

fig.suptitle("Figure 5c — Practical integration matrix", fontsize=13, weight="bold")
fig.tight_layout()
finish(fig, "figure5c_matrix")

raw.to_csv(TAB_DIR / "matrix_raw_counts.csv")
derived.to_csv(TAB_DIR / "matrix_derived.csv")
PAPER_MATRIX.to_csv(TAB_DIR / "matrix_published.csv")

agree = int((derived == PAPER_MATRIX).sum().sum())
print(f"\nCells agreeing with the published ratings: {agree} of {derived.size}")
if not IS_DEMO:
    d = diff.stack().sort_values()
    print("\nLargest divergences (worth re-reading, in either direction):")
    for (t, c), v in list(d.items())[:3] + list(d.items())[-3:]:
        if v != 0:
            print(f"  {v:+d}  {t[:44]:46s} x {c}")

---
## D5 — Run summary

In [ ]:
import hashlib

def sha(p, n=16):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()[:n]


outs = sorted(p for p in OUT_DIR.rglob("*") if p.is_file())
manifest = {
    "paper": {"title": "Artificial intelligence adoption in urban planning governance",
              "authors": "Lartey, D. & Law, K.M.Y.",
              "journal": "Landscape and Urban Planning 258 (2025) 105337",
              "doi": "10.1016/j.landurbplan.2025.105337"},
    "run": {"corpus_records": int(len(corpus)),
            "corpus_is_demo": bool(IS_DEMO),
            "network_is_demo": bool(VOS_IS_DEMO),
            "network_terms": int(len(items)), "network_links": int(len(links)),
            "clusters": int(items["cluster"].nunique()), "seed": SEED},
    "paper_targets": {"records": 588, "countries": 83, "retrieved": 3715},
    "outputs": {str(p.relative_to(OUT_DIR)): {"bytes": p.stat().st_size,
                                              "sha256_16": sha(p)} for p in outs},
}
path = OUT_DIR / "run_report.json"
path.write_text(json.dumps(manifest, indent=2, default=str))

print("=" * 64)
print("RUN COMPLETE")
print("=" * 64)
print(f"  corpus       {len(corpus):,} records" + ("  (DEMO)" if IS_DEMO else ""))
print(f"  network      {len(items):,} terms, {items['cluster'].nunique()} clusters"
      + ("  (SYNTHETIC)" if VOS_IS_DEMO else ""))
print(f"  outputs      {len(outs)} files in {OUT_DIR.resolve()}")
print("=" * 64)
for p in outs:
    print(f"  {str(p.relative_to(OUT_DIR)):46s} {p.stat().st_size / 1024:8.1f} KB")

---

### Citation

```bibtex
@article{lartey2025ai,
  title   = {Artificial intelligence adoption in urban planning governance:
             A systematic review of advancements in decision-making,
             and policy making},
  author  = {Lartey, Desmond and Law, Kris M. Y.},
  journal = {Landscape and Urban Planning},
  volume  = {258},
  pages   = {105337},
  year    = {2025},
  doi     = {10.1016/j.landurbplan.2025.105337}
}
```

Contact: **larteydesmond3@gmail.com**